# Where does Brazil's Congress spend its allowance?

*A first look at 2025 parliamentary expense data*

Brazil's federal deputies get a monthly allowance, called the **CEAP** (*Cota para o Exercício da Atividade Parlamentar*), to cover costs of doing the job: travel, office upkeep, communications, and so on. Every CEAP expense record is published as open data. In this notebook I take a first pass at that data, asking a simple question: **what does a "typical" expense charged to the quota look like?**

That turns out to be trickier than it sounds, and it's a nice way to see why the mean, median, and mode can tell very different stories.

**Source:** the Chamber of Deputies' [open data portal](https://dadosabertos.camara.leg.br/). The notebook downloads [`Ano-2025.csv.zip`](https://www.camara.leg.br/cotas/Ano-2025.csv.zip) straight from the Chamber's site, so there's nothing to download by hand. Field definitions come from the Chamber's [CEAP documentation](https://dadosabertos.camara.leg.br/howtouse/2023-12-26-dados-ceap.html).

**A note on the raw file:** it contains deputies' tax IDs (`cpf`) and passenger names on flight tickets. The notebook keeps the download in memory and never saves it to disk, so none of that ends up in this repository. The analysis below focuses on aggregate statistics and visualizations, and never displays those personal columns.

**The plan:**
1. Load the data and take a look.
2. Pick one numeric column and calculate its mean, median, and mode with pandas.
3. Redo the same calculations the hard way, using only Python's standard library.
4. Draw a simple chart, again using only the standard library.

## 1. Loading and looking at the data

The download is a zip file, which pandas can read directly. The CSV inside uses semicolons instead of commas as separators, and it's saved as UTF-8 with a "BOM" marker at the start, which is why the `sep` and `encoding` options are set below. (Without `utf-8-sig`, the first column name would get a hidden stray character stuck to it.)

In [1]:
import calendar
import io
import math
import urllib.request

import pandas as pd

DATA_URL = "https://www.camara.leg.br/cotas/Ano-2025.csv.zip"


def download(url):
    """Download a URL and return its contents as bytes."""
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        return response.read()


zip_bytes = download(DATA_URL)
expenses = pd.read_csv(
    io.BytesIO(zip_bytes),
    compression="zip",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False,
)

print(f"{len(expenses):,} rows, {len(expenses.columns)} columns")

209,086 rows, 32 columns


Each row represents an expense record submitted under the CEAP. Some supporting documents may be associated with multiple records, as explained in the Chamber's documentation.

In [2]:
COLUMNS_TO_SHOW = ["sgPartido", "sgUF", "txtDescricao", "numMes", "vlrLiquido"]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
expenses[COLUMNS_TO_SHOW].head()

,sgPartido,sgUF,txtDescricao,numMes,vlrLiquido
0,NaN,NaN,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,2,1467.0
1,NaN,NaN,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,3,1467.0
2,NaN,NaN,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,4,1467.0
3,NaN,NaN,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,5,1467.0
4,NaN,NaN,MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE ...,6,1467.0


Here's what those columns mean (based on the Brazilian Chamber of Deputies' official documentation):

- `sgPartido`: the party to which the deputy is affiliated (or was affiliated).
- `sgUF`: the state or Federal District through which the deputy was elected.
- `txtDescricao`: the category of expense.
- `numMes`: the financial competence month of the supporting document, determining the period in which the expense affects the CEAP.
- `vlrLiquido`: the net expense effectively debited from the parliamentary quota, in Brazilian reais (R$). This is the amount I'll analyze.

The first rows above have no party or state. Let me see how common that is, and who those rows belong to:

In [3]:
no_party_or_state = expenses["sgPartido"].isna() & expenses["sgUF"].isna()
amounts = expenses["vlrLiquido"]

print(
    f"{no_party_or_state.sum():,} rows "
    f"({no_party_or_state.mean():.1%}) have no party or state"
)
print(
    f"They make up "
    f"{amounts[no_party_or_state].sum() / amounts.sum():.2%} "
    f"of total spending"
)

expenses.loc[no_party_or_state, "txNomeParlamentar"].value_counts().head()

937 rows (0.4%) have no party or state
They make up 0.47% of total spending


txNomeParlamentar
LID.GOV-CD           284
LIDERANÇA DO PT      254
LIDERANÇA DO PSD      80
LIDERANÇA DO PDT      58
LIDERANÇA DO PSDB     53
Name: count, dtype: int64

Their `txNomeParlamentar` values suggest that these rows correspond to party leadership offices rather than individual deputies. The Chamber's documentation confirms that this field can contain the identifying title of a party leadership office, which can receive allocations from individual deputies' quotas. Since they are such a small share of both rows and spending, I'm leaving them in.

### A quick data check

Before trusting the numbers, I want to know what I'm dealing with. Are there any odd values in `vlrLiquido`?

In [4]:
print(f"Negative amounts: {(amounts < 0).sum():,}")
print(f"Zero amounts:     {(amounts == 0).sum():,}")
print(f"Smallest: R$ {amounts.min():,.2f}")
print(f"Largest:  R$ {amounts.max():,.2f}")

Negative amounts: 8,037
Zero amounts:     51
Smallest: R$ -61,932.80
Largest:  R$ 140,000.00


There are some negative amounts. Which categories do they come from?

In [5]:
expenses.loc[amounts < 0, "txtDescricao"].value_counts().head()

txtDescricao
PASSAGEM AÉREA - SIGEPA              7167
COMPLEMENTAÇÃO DO AUXÍLIO-MORADIA     834
PASSAGEM AÉREA - RPA                   35
PASSAGEM AÉREA - MISSÃO OFICIAL         1
Name: count, dtype: int64

The Chamber's documentation explains that negative values on airline tickets happen when a *compensation* ticket is issued for a ticket that wasn't used. That covers most of the negative rows. The documentation doesn't explain the other categories that show up here, so I'll treat them as adjustments too, but that's an assumption.

I'm keeping all negative amounts in, since they are part of the recorded financial adjustments and affect the net amount debited from the quota. Removing them would nudge the results. I use exactly the same rows in the pandas and standard-library versions below, so the two approaches stay comparable.

Next, the dates. My first idea was to chart spending by the date on each receipt (`datEmissao`). Let me check that it covers the year I expect:

In [6]:
issue_dates = pd.to_datetime(expenses["datEmissao"])

print(f"Earliest receipt date: {issue_dates.min().date()}")
print(f"Latest receipt date:   {issue_dates.max().date()}")
years_in_file = [int(year) for year in sorted(expenses["numAno"].unique())]
print(f"Years in the 'numAno' column: {years_in_file}")

missing_dates = issue_dates.isna().sum()
outside_year = (issue_dates.notna() & (issue_dates.dt.year != expenses["numAno"])).sum()

print(f"No receipt date: {missing_dates:,} rows ({missing_dates / len(expenses):.1%})")
print(
    f"Date outside file's year: {outside_year:,} rows "
    f"({outside_year / len(expenses):.1%})"
)
print(f"Rows with no month (numMes): {expenses['numMes'].isna().sum():,}")

Earliest receipt date: 2024-02-18
Latest receipt date:   2026-03-31
Years in the 'numAno' column: [2025]
No receipt date: 8,675 rows (4.1%)
Date outside file's year: 1,007 rows (0.5%)
Rows with no month (numMes): 0


**Dead end.** Some receipt dates fall outside the file's year, but the bigger limitation is the rows that have no receipt date at all (both counts are printed above). Since `numMes` is populated for every row (see the last count above), I'll use it for the monthly chart instead. According to the Chamber's documentation, `numMes` is the financial competence month of the supporting document (the period in which the expense affects the CEAP), and `numAno` is the corresponding year. That makes these fields more appropriate for analyzing spending by financial period than the receipt issuance date.

## 2. The mean, median, and mode with pandas

Now the main question. Pandas has a method for each of these.

In [7]:
pandas_mean = amounts.mean()
pandas_median = amounts.median()
pandas_mode = amounts.mode().tolist()  # a list, in case several values tie

print(f"Mean:   R$ {pandas_mean:,.2f}")
print(f"Median: R$ {pandas_median:,.2f}")
print(f"Mode:   {pandas_mode}")

Mean:   R$ 1,149.51
Median: R$ 252.87
Mode:   [200.0]


The three numbers disagree quite a bit, so what's going on?

- The **median** is the middle expense record: half of the records have smaller values, and half have larger values.
- The **mean** is the total divided by the number of records. A few very large expenses drag it upward.
- The **mode** is simply the most common exact amount across all records.

Let me put numbers on the gap, instead of just eyeballing it:

In [8]:
mean_to_median_ratio = pandas_mean / pandas_median
share_below_mean = (amounts < pandas_mean).mean()

print(f"The mean is {mean_to_median_ratio:.1f}x the median.")
print(f"{share_below_mean:.0%} of expense records have values below the mean.")

The mean is 4.5x the median.
78% of expense records have values below the mean.


So most expense records are well below the mean, and a minority of large ones pulls the mean up. Data like this is called **right-skewed**. If I'd only reported the mean, I'd have given a misleading picture of a typical expense. The median is a better description of a "typical" expense record here.

## 3. The same calculations, the hard way

Now I'll redo the exact same calculations without pandas or the `statistics` module, using Python's built-in `urllib`, `zipfile` and `csv` modules and basic loops. This cell is self-contained: it imports what it needs, downloads the file again itself, and doesn't use the DataFrame from above.

How each statistic is computed by hand:
- **Mean:** add everything up, divide by the count.
- **Median:** sort the values and take the middle one (or the average of the two middle ones when the count is even).
- **Mode:** count how often each value appears in a dictionary, then find the value(s) with the highest count.

In [9]:
# These are imported here on purpose, so this cell works on its own
import csv
import io
import urllib.request
import zipfile

DATA_URL = "https://www.camara.leg.br/cotas/Ano-2025.csv.zip"
COLUMN_NAME = "vlrLiquido"


def read_column(url, column_name):
    """Read one numeric column from the zipped, semicolon-delimited CSV."""
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        zip_bytes = response.read()

    values = []
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        csv_name = next(name for name in archive.namelist() if name.endswith(".csv"))
        with archive.open(csv_name) as raw_file:
            text_file = io.TextIOWrapper(raw_file, encoding="utf-8-sig", newline="")
            for row in csv.DictReader(text_file, delimiter=";"):
                cell = row[column_name].strip()
                if cell != "":  # skip empty cells
                    values.append(float(cell))
    return values


def calculate_mean(values):
    """Return the mean: the sum of the values divided by how many there are."""
    total = 0.0
    for value in values:
        total += value
    return total / len(values)


def calculate_median(values):
    """Return the middle value (or the average of the two middle values)."""
    sorted_values = sorted(values)
    middle = len(sorted_values) // 2
    if len(sorted_values) % 2 == 1:
        return sorted_values[middle]
    return (sorted_values[middle - 1] + sorted_values[middle]) / 2


def calculate_mode(values):
    """Return a sorted list of the most common value(s) and how often they appear."""
    counts = {}
    for value in values:
        counts[value] = counts.get(value, 0) + 1

    highest_count = max(counts.values())
    modes = [value for value, count in counts.items() if count == highest_count]
    return sorted(modes), highest_count


amount_values = read_column(DATA_URL, COLUMN_NAME)
hard_mean = calculate_mean(amount_values)
hard_median = calculate_median(amount_values)
hard_mode, mode_count = calculate_mode(amount_values)

print(f"Rows read: {len(amount_values):,}")
print(f"Mean:   R$ {hard_mean:,.2f}")
print(f"Median: R$ {hard_median:,.2f}")
print(f"Mode:   {hard_mode} (appears {mode_count:,} times)")

Rows read: 209,086
Mean:   R$ 1,149.51
Median: R$ 252.87
Mode:   [200.0] (appears 5,103 times)


If both approaches are right, they should agree. Let me check that:

In [10]:
print("Mean matches pandas:  ", math.isclose(pandas_mean, hard_mean))
print("Median matches pandas:", math.isclose(pandas_median, hard_median))
print("Mode matches pandas:  ", pandas_mode == hard_mode)

Mean matches pandas:   True
Median matches pandas: True
Mode matches pandas:   True


They agree. One more question: what are the expense records at the most common amount actually for?

In [11]:
most_common_amount = pandas_mode[0]
amounts_at_mode = expenses[amounts == most_common_amount]
category_shares = amounts_at_mode["txtDescricao"].value_counts(normalize=True)

print(f"Expense records of exactly R$ {most_common_amount:,.2f}, by category:")
print(category_shares.head(3).map("{:.1%}".format))

Expense records of exactly R$ 200.00, by category:
txtDescricao
COMBUSTÍVEIS E LUBRIFICANTES.                                95.1%
HOSPEDAGEM ,EXCETO DO PARLAMENTAR NO DISTRITO FEDERAL.        2.5%
MANUTENÇÃO DE ESCRITÓRIO DE APOIO À ATIVIDADE PARLAMENTAR     0.9%
Name: proportion, dtype: object


One category accounts for the overwhelming majority of expense records at that exact amount. This suggests that the mode is largely driven by that one top category rather than representing a typical expense across all categories. I don't know why this amount repeats, so that's an interesting question for follow-up.

## 4. A simple chart: spending by month

To see how spending changes across the year, I'll add up everything charged to the quota per month. The totals are calculated with pandas, but the drawing code afterward uses nothing but the standard library: just text and string repetition. Each `█` represents a fixed chunk of money, so a longer bar means more spending.

In [12]:
monthly_totals = expenses.groupby("numMes")["vlrLiquido"].sum()
monthly_millions = {
    int(month): total / 1_000_000 for month, total in monthly_totals.items()
}

years_in_data = sorted(expenses["numAno"].unique())
year_label = ", ".join(str(year) for year in years_in_data)

In [13]:
# Drawing code: standard library only
MAX_BAR_WIDTH = 40  # characters; keeps the chart readable on a narrow screen

biggest_month_total = max(monthly_millions.values())
millions_per_block = biggest_month_total / MAX_BAR_WIDTH

print(f"CEAP spending charged to the quota by month, {year_label} (R$ millions)")
print(f"Each █ is about R$ {millions_per_block:.2f} million")
print()

for month in sorted(monthly_millions):
    month_total = monthly_millions[month]
    bar = "█" * round(month_total / millions_per_block)
    print(f"{calendar.month_abbr[month]} | {bar} {month_total:.1f}")

CEAP spending charged to the quota by month, 2025 (R$ millions)
Each █ is about R$ 0.57 million

Jan | ███████████████████████████████ 17.7
Feb | █████████████████████████████████████ 21.4
Mar | █████████████████████████████████████ 21.3
Apr | ███████████████████████████████████████ 22.3
May | ██████████████████████████████████████ 21.9
Jun | ██████████████████████████████████████ 21.7
Jul | ████████████████████████████████████████ 22.9
Aug | ███████████████████████████████ 18.0
Sep | ████████████████████████████████ 18.3
Oct | █████████████████████████████████ 18.7
Nov | ███████████████████████████████ 17.9
Dec | ████████████████████████████████ 18.2


Let me back up what the bars suggest with a few numbers, calculated from the same monthly totals:

In [14]:
monthly_average = sum(monthly_millions.values()) / len(monthly_millions)
peak_month = max(monthly_millions, key=monthly_millions.get)
lowest_month = min(monthly_millions, key=monthly_millions.get)
months_above_average = [
    calendar.month_abbr[month]
    for month in sorted(monthly_millions)
    if monthly_millions[month] > monthly_average
]

print(f"Average month: R$ {monthly_average:.1f} million")
print(
    f"Highest: {calendar.month_abbr[peak_month]} "
    f"(R$ {monthly_millions[peak_month]:.1f} million)"
)
print(
    f"Lowest:  {calendar.month_abbr[lowest_month]} "
    f"(R$ {monthly_millions[lowest_month]:.1f} million)"
)
print(f"Months above average: {', '.join(months_above_average)}")

Average month: R$ 20.0 million
Highest: Jul (R$ 22.9 million)
Lowest:  Jan (R$ 17.7 million)
Months above average: Feb, Mar, Apr, May, Jun, Jul


**What I take from this:** in the 2025 file, the months above average sit together in one block rather than scattering through the year, suggesting two distinct levels of spending: a sustained stretch of higher monthly totals, and lower totals outside it. The reasons for the variation are not explored here, so the pattern should be treated as a descriptive finding rather than an explanation of spending behavior.

## Caveats and next steps

- **This is spending charged to the quota, not everything deputies spend.** The dataset only includes expenses that were recorded against the CEAP.
- **Negative amounts are included as-is.** Dropping them would change the totals and statistics. I kept them because they're part of the real record.
- **The data is downloaded fresh on every run.** If the Chamber corrects or adds records, the numbers will shift slightly. That's why the figures in the text are printed by code rather than typed in.
- **One year only.** I can't say whether 2025 is typical without comparing it to other years.
- **No scientific claims here.** I've described how the numbers are distributed, not tested why.
- **Missing party and state information:** a small share of records (see section 1) have no party or state, and appear to be party leadership offices rather than individual deputies. They are included, but they account for a tiny share of total spending and are unlikely to materially affect the aggregate results.

Next, I'd look at *which categories* (`txtDescricao`) and *which parties or states* drive the large expenses that skew the mean.